In [ ]:
from pathlib import Path
import json
import re
from typing import Any, Dict, List, Optional
import soundfile as sf
import traceback

import pandas as pd
from tqdm.auto import tqdm

import torch
from qwen_asr import Qwen3ASRModel

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3"

In [ ]:
# =========================
# Config
# =========================

AUDIO_DIR = Path("/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios")
OUTPUT_DIR = Path("/mnt/ssd/hbli/songformer/runs/qwen_lyrics/harmonixset/")

ASR_MODEL_PATH = "/home/hbli/songformer/repo/SongFormer/src/third_party/Qwen3-ASR-1.7B"
FORCED_ALIGNER_PATH = "/home/hbli/songformer/repo/SongFormer/src/third_party/Qwen3-ForcedAligner-0.6B"

# 1-2 case
MAX_FILES = 2

FORCE_LANGUAGE = "English"

# inference params
MAX_INFERENCE_BATCH_SIZE = 4
MAX_NEW_TOKENS = 1024

# ForcedAligner: up to 5 minutes of speech
MAX_AUDIO_SEC_FOR_ALIGNMENT = 300.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(AUDIO_DIR)
print(OUTPUT_DIR)

/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios
/mnt/ssd/hbli/songformer/runs/qwen_lyrics/harmonixset


In [5]:
def safe_get(obj: Any, key: str, default=None):
    if obj is None:
        return default
    if isinstance(obj, dict):
        return obj.get(key, default)
    return getattr(obj, key, default)


def to_builtin(obj: Any):
    """
    把复杂对象尽量转成 Python 原生类型，方便 json.dump
    """
    if obj is None:
        return None
    if isinstance(obj, (str, int, float, bool)):
        return obj
    if isinstance(obj, dict):
        return {k: to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [to_builtin(v) for v in obj]
    if hasattr(obj, "__dict__"):
        return {k: to_builtin(v) for k, v in vars(obj).items()}
    return str(obj)


def maybe_float(x):
    if x is None:
        return None
    try:
        return float(x)
    except Exception:
        return x


def get_audio_duration_sec(audio_path: Path) -> float:
    info = sf.info(str(audio_path))
    return float(info.frames) / float(info.samplerate)


def flatten_alignment_units(time_stamps_obj: Any) -> List[Dict[str, Any]]:
    """
    把 result.time_stamps 尽量拍平成一个 list。
    目标是提取最小对齐单元：text/start_time/end_time
    对英文歌通常会接近 word-level。
    """
    flattened = []

    def _walk(x):
        if x is None:
            return

        if isinstance(x, dict):
            has_core = (
                ("text" in x or "token" in x or "word" in x or "char" in x)
                and ("start_time" in x or "start" in x)
                and ("end_time" in x or "end" in x)
            )
            if has_core:
                flattened.append(
                    {
                        "text": x.get("text") or x.get("token") or x.get("word") or x.get("char") or "",
                        "start_time": maybe_float(x.get("start_time", x.get("start"))),
                        "end_time": maybe_float(x.get("end_time", x.get("end"))),
                    }
                )
                return

            for v in x.values():
                _walk(v)
            return

        if isinstance(x, (list, tuple)):
            for v in x:
                _walk(v)
            return

        text = safe_get(x, "text", None)
        if text is None:
            text = safe_get(x, "token", None)
        if text is None:
            text = safe_get(x, "word", None)
        if text is None:
            text = safe_get(x, "char", None)

        start = safe_get(x, "start_time", safe_get(x, "start", None))
        end = safe_get(x, "end_time", safe_get(x, "end", None))

        if text is not None and start is not None and end is not None:
            flattened.append(
                {
                    "text": text,
                    "start_time": maybe_float(start),
                    "end_time": maybe_float(end),
                }
            )
            return

        if hasattr(x, "__dict__"):
            for v in vars(x).values():
                _walk(v)

    _walk(time_stamps_obj)

    flattened = sorted(
        flattened,
        key=lambda d: (
            float("inf") if d["start_time"] is None else d["start_time"],
            float("inf") if d["end_time"] is None else d["end_time"],
        )
    )
    return flattened


def normalize_asr_result(result_obj: Any) -> Dict[str, Any]:
    """
    统一抽取：
    - language
    - raw_transcript
    - aligned_words（或更准确地说 aligned units）
    - raw time_stamps
    """
    language = safe_get(result_obj, "language", "")
    raw_transcript = safe_get(result_obj, "text", "")
    time_stamps_raw = safe_get(result_obj, "time_stamps", None)

    aligned_units = flatten_alignment_units(time_stamps_raw)

    return {
        "language": language,
        "raw_transcript": raw_transcript,
        "aligned_words": aligned_units,   # 英文 case 下预期主要是 word-level
        "time_stamps_raw": to_builtin(time_stamps_raw),
        "raw_result": to_builtin(result_obj),
    }


def save_lyrics_json(audio_path: Path, norm_result: Dict[str, Any], output_dir: Path):
    """
    输出文件名与 audio 完全同 stem，仅扩展名改为 .json
    比如:
    xxx.wav -> xxx.json
    """
    output_path = output_dir / f"{audio_path.stem}.json"

    payload = {
        "song_id": audio_path.stem,
        "audio_filename": audio_path.name,
        "audio_path": str(audio_path),
        "language": norm_result["language"],
        "raw_transcript": norm_result["raw_transcript"],
        "aligned_words": norm_result["aligned_words"],
        "time_stamps_raw": norm_result["time_stamps_raw"],
    }

    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(payload, f, ensure_ascii=False, indent=2)

    return output_path

In [6]:
model = Qwen3ASRModel.from_pretrained(
    ASR_MODEL_PATH,
    dtype=torch.bfloat16,
    device_map="auto",
    max_inference_batch_size=MAX_INFERENCE_BATCH_SIZE,
    max_new_tokens=MAX_NEW_TOKENS,
    forced_aligner=FORCED_ALIGNER_PATH,
    forced_aligner_kwargs=dict(
        dtype=torch.bfloat16,
        device_map="auto",
    ),
)

We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).
Loading checkpoint shards: 100%|██████████| 2/2 [00:02<00:00,  1.05s/it]
The following generation flags are not valid and may be ignored: ['temperature']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
We will use 90% of the memory on device 0 for storing the model, and 10% for the buffer to avoid OOM. You can set `max_memory` in to a higher value to use more memory (at your own risk).


#### transcribe test

In [13]:
wav_files = sorted(AUDIO_DIR.glob("*.wav"))
test_files = wav_files[0]
test_files

PosixPath('/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios/HX_0003_6foot7foot.wav')

In [14]:
duration_sec = get_audio_duration_sec(test_files)
duration_sec

157.2455328798186

In [15]:
results = model.transcribe(
    audio=str(test_files),
    language=None,
    return_time_stamps=True,
)

Setting `pad_token_id` to `eos_token_id`:151645 for open-end generation.


In [31]:
time_stamps = getattr(results[0], 'time_stamps')

In [32]:
flatten_alignment_units(time_stamps)

[{'text': 'Oh', 'start_time': 0.0, 'end_time': 0.0},
 {'text': "it's", 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'killing', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'my', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'charisma', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'I', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'call', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'it', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'pressure', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'swagger', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'bad', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'Call', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'it', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'ambition', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'young', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'money', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'ambition', 'start_time': 0.0, 'end_time': 0.0},
 {'text': 'and', 'start_time': 0.0, 'end_time': 0.0},

#### output test

In [ ]:
def transcribe_one(audio_path: Path, model: Qwen3ASRModel, language: Optional[str] = None) -> Dict[str, Any]:
    duration_sec = get_audio_duration_sec(audio_path)

    if duration_sec > MAX_AUDIO_SEC_FOR_ALIGNMENT:
        raise RuntimeError(
            f"Audio too long for current alignment setting: {duration_sec:.2f}s > "
            f"{MAX_AUDIO_SEC_FOR_ALIGNMENT:.2f}s"
        )

    results = model.transcribe(
        audio=str(audio_path),
        language=language,
        return_time_stamps=True,
    )

    if results is None or len(results) == 0:
        raise RuntimeError(f"Empty result for {audio_path.name}")

    result_obj = results[0]
    norm_result = normalize_asr_result(result_obj)
    norm_result["audio_duration_sec"] = duration_sec

    return norm_result

In [37]:
wav_files = sorted(AUDIO_DIR.glob("*.wav"))
test_files = wav_files[:3]
test_files

[PosixPath('/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios/HX_0003_6foot7foot.wav'),
 PosixPath('/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios/HX_0006_aint2proud2beg.wav'),
 PosixPath('/mnt/ssd/hbli/datasets/songformer/songformdb/HX/audios/HX_0008_america.wav')]

In [40]:
run_log = []

for audio_path in tqdm(test_files, desc="Qwen3-ASR + aligner"):
    try:
        norm_result = transcribe_one(
            audio_path=audio_path,
            model=model,
            language=FORCE_LANGUAGE,
        )

        output_path = save_lyrics_json(
            audio_path=audio_path,
            norm_result=norm_result,
            output_dir=OUTPUT_DIR,
        )

        run_log.append(
            {
                "audio": audio_path.name,
                "status": "ok",
                "audio_duration_sec": norm_result["audio_duration_sec"],
                "language": norm_result["language"],
                "num_aligned_words": len(norm_result["aligned_words"]),
                "output_json": str(output_path),
            }
        )

    except Exception as e:
        run_log.append(
            {
                "audio": audio_path.name,
                "status": "error",
                "error": str(e),
                "traceback": traceback.format_exc(),
            }
        )

print(json.dumps(run_log, ensure_ascii=False, indent=2))

Qwen3-ASR + aligner: 100%|██████████| 3/3 [02:27<00:00, 49.15s/it]

[
  {
    "audio": "HX_0003_6foot7foot.wav",
    "status": "ok",
    "audio_duration_sec": 157.2455328798186,
    "language": "English",
    "num_aligned_words": 427,
    "output_json": "/mnt/ssd/hbli/songformer/runs/qwen_lyrics/harmonixset/HX_0003_6foot7foot.json"
  },
  {
    "audio": "HX_0006_aint2proud2beg.wav",
    "status": "ok",
    "audio_duration_sec": 180.88344671201813,
    "language": "English",
    "num_aligned_words": 653,
    "output_json": "/mnt/ssd/hbli/songformer/runs/qwen_lyrics/harmonixset/HX_0006_aint2proud2beg.json"
  },
  {
    "audio": "HX_0008_america.wav",
    "status": "ok",
    "audio_duration_sec": 222.49360544217686,
    "language": "English",
    "num_aligned_words": 117,
    "output_json": "/mnt/ssd/hbli/songformer/runs/qwen_lyrics/harmonixset/HX_0008_america.json"
  }
]
